## ORDER & PRODUCT & SESSION

In [18]:
import pandas as pd
import numpy as np
from datetime import datetime

# Import ORDER, PRODUCT and SESSION files
order = pd.read_csv('../DataSet/ORDER.csv')
product = pd.read_csv('../DataSet/PRODUCT.csv')
session = pd.read_csv('../DataSet/SESSION.csv')

print("=" * 80)
print("BEFORE CLEANING - DATA OVERVIEW")
print("=" * 80)
print("\nORDER Dataset:")
print(f"Shape: {order.shape}")
print(f"Missing values: \n{order.isnull().sum()}")
print(order.head())

print("\n\nPRODUCT Dataset:")
print(f"Shape: {product.shape}")
print(f"Missing values: \n{product.isnull().sum()}")
print(product.head())

print("\n\nSESSION Dataset:")
print(f"Shape: {session.shape}")
print(f"Missing values: \n{session.isnull().sum()}")
print(session.head())

BEFORE CLEANING - DATA OVERVIEW

ORDER Dataset:
Shape: (356242, 8)
Missing values: 
order_id        0
customer_id     0
order_date      0
store_id        0
channel_id      0
payment_type    0
year            0
week            0
dtype: int64
      order_id customer_id  order_date store_id channel_id      payment_type  \
0  ORD00000001  CUST001477  16-05-2025   ST0049      CH002       Credit Card   
1  ORD00000002  CUST006101  09-01-2024   ST0017      CH001        Debit Card   
2  ORD00000003  CUST004107  02-09-2025   ST0081      CH002               UPI   
3  ORD00000004  CUST006214  26-04-2025   ST0086      CH003  Cash on Delivery   
4  ORD00000005  CUST000101  18-07-2025   ST0021      CH001        Debit Card   

   year  week  
0  2025    20  
1  2024     2  
2  2025    36  
3  2025    17  
4  2025    29  


PRODUCT Dataset:
Shape: (200, 4)
Missing values: 
product_id     0
category_id    0
brand          0
sku_name       0
dtype: int64
  product_id category_id    brand  \
0  PROD00001

In [19]:
# 1. CLEAN ORDER DATASET
print("\n" + "=" * 80)
print("CLEANING: ORDER DATASET")
print("=" * 80)

# Create a copy to preserve original
order_clean = order.copy()

# Convert order_date from DD-MM-YYYY string to datetime
print("\n1. Converting order_date from DD-MM-YYYY string to datetime...")
order_clean['order_date'] = pd.to_datetime(order_clean['order_date'], format='%d-%m-%Y')
print(f"   ✓ Converted order_date to datetime. Sample:\n{order_clean['order_date'].head()}")

# Recalculate year and week based on actual order_date
print("\n2. Recalculating year and week columns based on actual order_date...")
order_clean['year'] = order_clean['order_date'].dt.isocalendar().year
order_clean['week'] = order_clean['order_date'].dt.isocalendar().week
print(f"   ✓ Updated year and week columns. Sample values:")
print(order_clean[['order_date', 'year', 'week']].head(10))

# Check for duplicate order_ids
print("\n3. Checking for duplicate order_ids...")
duplicates = order_clean[order_clean.duplicated(subset=['order_id'], keep=False)]
print(f"   • Duplicate records found: {len(duplicates)}")
if len(duplicates) > 0:
    print(f"   • Removing {len(duplicates)//2} duplicate pairs...")
    order_clean = order_clean.drop_duplicates(subset=['order_id'], keep='first')
    print(f"   ✓ Duplicates removed. New shape: {order_clean.shape}")
else:
    print("   ✓ No duplicates found")

# Verify data types
print("\n4. Verifying data types...")
print(order_clean.dtypes)

# Summary statistics
print("\n5. Final ORDER dataset statistics:")
print(f"   • Total records: {len(order_clean)}")
print(f"   • Columns: {list(order_clean.columns)}")
print(f"   • Missing values:\n{order_clean.isnull().sum()}")
print(f"   • Date range: {order_clean['order_date'].min()} to {order_clean['order_date'].max()}")
print(f"   • Payment types: {order_clean['payment_type'].unique().tolist()}")
print(f"   • Number of stores: {order_clean['store_id'].nunique()}")
print(f"   • Number of channels: {order_clean['channel_id'].nunique()}")
print("\n✓ ORDER dataset cleaning complete!")


CLEANING: ORDER DATASET

1. Converting order_date from DD-MM-YYYY string to datetime...
   ✓ Converted order_date to datetime. Sample:
0   2025-05-16
1   2024-01-09
2   2025-09-02
3   2025-04-26
4   2025-07-18
Name: order_date, dtype: datetime64[ns]

2. Recalculating year and week columns based on actual order_date...
   ✓ Updated year and week columns. Sample values:
  order_date  year  week
0 2025-05-16  2025    20
1 2024-01-09  2024     2
2 2025-09-02  2025    36
3 2025-04-26  2025    17
4 2025-07-18  2025    29
5 2025-09-18  2025    38
6 2024-10-03  2024    40
7 2024-04-04  2024    14
8 2023-08-23  2023    34
9 2025-03-09  2025    10

3. Checking for duplicate order_ids...
   • Duplicate records found: 0
   ✓ No duplicates found

4. Verifying data types...
order_id                object
customer_id             object
order_date      datetime64[ns]
store_id                object
channel_id              object
payment_type            object
year                    UInt32
week       

In [20]:
# 2. CLEAN PRODUCT DATASET
print("\n" + "=" * 80)
print("CLEANING: PRODUCT DATASET")
print("=" * 80)

product_clean = product.copy()

# Check for duplicates
print("\n1. Checking for duplicate product_ids...")
duplicates = product_clean[product_clean.duplicated(subset=['product_id'], keep=False)]
print(f"   • Duplicate records found: {len(duplicates)}")
if len(duplicates) > 0:
    print(f"   • Removing duplicates...")
    product_clean = product_clean.drop_duplicates(subset=['product_id'], keep='first')
    print(f"   ✓ Duplicates removed. New shape: {product_clean.shape}")
else:
    print("   ✓ No duplicates found")

# Trim whitespace from all string columns
print("\n2. Trimming whitespace from string columns...")
for col in product_clean.select_dtypes(include=['object']).columns:
    product_clean[col] = product_clean[col].str.strip()
print("   ✓ Whitespace trimmed")

# Standardize case for brand names
print("\n3. Standardizing brand names to Title Case...")
product_clean['brand'] = product_clean['brand'].str.title()
print(f"   ✓ Brands: {product_clean['brand'].unique().tolist()}")

# Check for missing values
print("\n4. Checking for missing values...")
missing = product_clean.isnull().sum()
print(f"   Missing values:\n{missing}")
if missing.sum() == 0:
    print("   ✓ No missing values found")
else:
    print(f"   ⚠ Found {missing.sum()} missing values - filling with 'Unknown'")
    product_clean = product_clean.fillna('Unknown')

# Summary statistics
print("\n5. Final PRODUCT dataset statistics:")
print(f"   • Total records: {len(product_clean)}")
print(f"   • Columns: {list(product_clean.columns)}")
print(f"   • Missing values:\n{product_clean.isnull().sum()}")
print(f"   • Unique brands: {product_clean['brand'].nunique()}")
print(f"   • Unique categories: {product_clean['category_id'].nunique()}")
print(f"   • Categories: {sorted(product_clean['category_id'].unique().tolist())}")
print("\n✓ PRODUCT dataset cleaning complete!")


CLEANING: PRODUCT DATASET

1. Checking for duplicate product_ids...
   • Duplicate records found: 0
   ✓ No duplicates found

2. Trimming whitespace from string columns...
   ✓ Whitespace trimmed

3. Standardizing brand names to Title Case...
   ✓ Brands: ['Nestle', 'Nescafe', 'Milo', 'Milkmaid', 'Nespray', 'Cerelac', 'Lactogen', 'Nan', 'Kitkat', 'Bar-One', 'Munch', 'Maggi', 'Nestum', 'Fitnesse', 'Honey Stars', 'Nesquik', 'Cheerios', 'Koko Krunch', 'Friskies', 'Felix', 'Purina', 'Pro Plan', 'Pure Life', 'Boost', 'Peptamen', 'Optifast', 'Resource']

4. Checking for missing values...
   Missing values:
product_id     0
category_id    0
brand          0
sku_name       0
dtype: int64
   ✓ No missing values found

5. Final PRODUCT dataset statistics:
   • Total records: 200
   • Columns: ['product_id', 'category_id', 'brand', 'sku_name']
   • Missing values:
product_id     0
category_id    0
brand          0
sku_name       0
dtype: int64
   • Unique brands: 27
   • Unique categories: 10
  

In [21]:
# 3. CLEAN SESSION DATASET
print("\n" + "=" * 80)
print("CLEANING: SESSION DATASET")
print("=" * 80)

session_clean = session.copy()

# Convert session_start from string to datetime
print("\n1. Converting session_start from string to datetime...")
session_clean['session_start'] = pd.to_datetime(session_clean['session_start'], format='%Y-%m-%d %H:%M:%S')
print(f"   ✓ Converted session_start to datetime. Sample:\n{session_clean['session_start'].head()}")

# Validate and clean numeric columns
print("\n2. Validating numeric columns...")
# Ensure session_duration_sec is positive
invalid_duration = session_clean[session_clean['session_duration_sec'] <= 0]
if len(invalid_duration) > 0:
    print(f"   ⚠ Found {len(invalid_duration)} sessions with invalid duration (≤0). Removing...")
    session_clean = session_clean[session_clean['session_duration_sec'] > 0]
else:
    print("   ✓ All session durations are positive")

# Ensure pages_viewed is positive integer
invalid_pages = session_clean[session_clean['pages_viewed'] <= 0]
if len(invalid_pages) > 0:
    print(f"   ⚠ Found {len(invalid_pages)} sessions with invalid pages_viewed (≤0). Removing...")
    session_clean = session_clean[session_clean['pages_viewed'] > 0]
else:
    print("   ✓ All pages_viewed are positive")

session_clean['pages_viewed'] = session_clean['pages_viewed'].astype('int64')
print("   ✓ Pages_viewed converted to integer")

# Check for duplicate session_ids
print("\n3. Checking for duplicate session_ids...")
duplicates = session_clean[session_clean.duplicated(subset=['session_id'], keep=False)]
print(f"   • Duplicate records found: {len(duplicates)}")
if len(duplicates) > 0:
    print(f"   • Removing duplicates...")
    session_clean = session_clean.drop_duplicates(subset=['session_id'], keep='first')
    print(f"   ✓ Duplicates removed. New shape: {session_clean.shape}")
else:
    print("   ✓ No duplicates found")

# Standardize device and referrer text
print("\n4. Standardizing device and referrer fields...")
session_clean['device'] = session_clean['device'].str.strip()
session_clean['referrer'] = session_clean['referrer'].str.strip()
print(f"   ✓ Devices: {sorted(session_clean['device'].unique().tolist())}")
print(f"   ✓ Referrers (sample): {session_clean['referrer'].unique()[:10].tolist()}")

# Check for missing values
print("\n5. Checking for missing values...")
missing = session_clean.isnull().sum()
print(f"   Missing values:\n{missing}")
if missing.sum() == 0:
    print("   ✓ No missing values found")
else:
    print(f"   ⚠ Found {missing.sum()} missing values")

# Summary statistics
print("\n6. Final SESSION dataset statistics:")
print(f"   • Total records: {len(session_clean)}")
print(f"   • Columns: {list(session_clean.columns)}")
print(f"   • Missing values:\n{session_clean.isnull().sum()}")
print(f"   • Date range: {session_clean['session_start'].min()} to {session_clean['session_start'].max()}")
print(f"   • Session duration range: {session_clean['session_duration_sec'].min()} to {session_clean['session_duration_sec'].max()} seconds")
print(f"   • Pages viewed range: {session_clean['pages_viewed'].min()} to {session_clean['pages_viewed'].max()}")
print(f"   • Unique devices: {session_clean['device'].nunique()}")
print(f"   • Unique referrers: {session_clean['referrer'].nunique()}")
print("\n✓ SESSION dataset cleaning complete!")


CLEANING: SESSION DATASET

1. Converting session_start from string to datetime...
   ✓ Converted session_start to datetime. Sample:
0   2025-05-17 17:22:05
1   2025-05-14 07:01:37
2   2025-05-16 11:55:51
3   2025-05-18 22:48:19
4   2025-05-14 12:39:26
Name: session_start, dtype: datetime64[ns]

2. Validating numeric columns...
   ✓ All session durations are positive
   ✓ All pages_viewed are positive
   ✓ Pages_viewed converted to integer

3. Checking for duplicate session_ids...
   • Duplicate records found: 0
   ✓ No duplicates found

4. Standardizing device and referrer fields...
   ✓ Devices: ['Desktop', 'Mobile', 'Tablet']
   ✓ Referrers (sample): ['Email', 'Direct', 'Affiliate', 'Google Search', 'Push', 'Instagram', 'Marketplace', 'YouTube']

5. Checking for missing values...
   Missing values:
session_id              0
customer_id             0
session_start           0
session_duration_sec    0
pages_viewed            0
device                  0
referrer                0
dtype

In [22]:
# 4. VALIDATE CLEANED DATASETS
print("\n" + "=" * 80)
print("DATA VALIDATION")
print("=" * 80)

# Validate data integrity
print("\nValidation Results:")

# ORDER validation
print("\n✓ ORDER Dataset:")
print(f"  - Total records: {len(order_clean):,}")
print(f"  - Date range: {order_clean['order_date'].min().date()} to {order_clean['order_date'].max().date()}")
print(f"  - All years valid: {order_clean['year'].min()}-{order_clean['year'].max()}")
print(f"  - All weeks in range [1-53]: {order_clean['week'].min()}-{order_clean['week'].max()}")
print(f"  - No missing values: {order_clean.isnull().sum().sum() == 0}")

# PRODUCT validation
print("\n✓ PRODUCT Dataset:")
print(f"  - Total records: {len(product_clean):,}")
print(f"  - All unique product_ids: {product_clean['product_id'].nunique() == len(product_clean)}")
print(f"  - Brands: {product_clean['brand'].nunique()}")
print(f"  - Categories: {product_clean['category_id'].nunique()}")
print(f"  - No missing values: {product_clean.isnull().sum().sum() == 0}")

# SESSION validation 
print("\n✓ SESSION Dataset:")
print(f"  - Total records: {len(session_clean):,}")
print(f"  - Date range: {session_clean['session_start'].min()} to {session_clean['session_start'].max()}")
print(f"  - All unique session_ids: {session_clean['session_id'].nunique() == len(session_clean)}")
print(f"  - Valid session durations: {(session_clean['session_duration_sec'] > 0).all()}")
print(f"  - Valid pages_viewed: {(session_clean['pages_viewed'] > 0).all()}")
print(f"  - No missing values: {session_clean.isnull().sum().sum() == 0}")

print("\n" + "=" * 80)
print("✓ All datasets validated successfully!")


DATA VALIDATION

Validation Results:

✓ ORDER Dataset:
  - Total records: 356,242
  - Date range: 2023-02-26 to 2026-02-21
  - All years valid: 2023-2026
  - All weeks in range [1-53]: 1-52
  - No missing values: True

✓ PRODUCT Dataset:
  - Total records: 200
  - All unique product_ids: True
  - Brands: 27
  - Categories: 10
  - No missing values: True

✓ SESSION Dataset:
  - Total records: 1,259,647
  - Date range: 2023-02-25 09:57:41 to 2026-02-23 22:59:41
  - All unique session_ids: True
  - Valid session durations: True
  - Valid pages_viewed: True
  - No missing values: True

✓ All datasets validated successfully!


# run this after finish all cleaning process

In [23]:
# Save cleaned datasets
order.to_csv('../Cleaned_DataSet/ORDER.csv', index=False)
product.to_csv('../Cleaned_DataSet/PRODUCT.csv', index=False)
session.to_csv('../Cleaned_DataSet/SESSION.csv', index=False)
print("Cleaned datasets saved to Cleaned_DataSet folder")

Cleaned datasets saved to Cleaned_DataSet folder
